# 生成AIでGISコードを書く — 実践ワークフロー

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-sat-gis/blob/main/public/notebooks/tutorials/00_ai_assisted_gis.ipynb)

このノートブックでは、生成AIをコーディングパートナーとして使いながら衛星データ解析のコードを書くワークフローを体験する。

### ワークフロー
```
問題定義 → AIにプロンプト → コード生成 → Colabに貼り付け → 実行 → 検証 → 必要ならAIに修正依頼
```

### 使用するAIツール（おすすめ）
- **Colab + Geminiサイドパネル**（環境構築不要）
- または **ChatGPT Free / Google Gemini** でコード生成 → Colabに貼り付け

### このノートブックの構成

| パート | 内容 | データソース |
|--------|------|-------------|
| A | NDVI計算（GEE） | Sentinel-2 |
| B | SAR後方散乱係数（GEE） | Sentinel-1 |
| C | 標高データ取得（JAXA Earth API） | ALOS AW3D30 |
| D | AIを使ったデバッグ練習 | — |

各セルに「💬 プロンプト例」と「🤖 AIの応答（例）」を記載している。実際に使うときは自分の言葉でAIに依頼してみよう。

---
## Part A: GEE + Sentinel-2 NDVI

### シナリオ
福島県浜通りの農地について、2024年6月のNDVIを計算したい。

### 💬 Gemini/ChatGPTへのプロンプト例
```
あなたはGEE（Google Earth Engine）のPython APIの専門家です。
以下のコードを生成してください。

- Sentinel-2 SR_HARMONIZED からNDVIを計算
- 範囲: ee.Geometry.Rectangle([139.9, 37.2, 140.6, 37.8])
- 期間: 2024-06-01 〜 2024-06-30
- 雲20%未満の画像のみ使用
- 最も雲の少ない画像を選んでNDVIを計算
- geemapで地図表示（RGB画像とNDVIを重ねる）
- NDVIの範囲（最小・最大）を表示する
```

In [ ]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
!pip install geemap earthengine-api japanize-matplotlib


In [ ]:
import ee
import geemap
try:
    import japanize_matplotlib  # あれば使う（無くても env.setup_font() が日本語対応）
except ImportError:
    japanize_matplotlib = None  # 未インストールでも続行（ローカルはシステムフォントで表示）
try:
    from env import setup_font
    setup_font()
except ImportError:
    pass

# GEE の認証と初期化（Colab / ローカル両対応）
try:
    from env import gee_setup
    gee_setup()
except ImportError:
    # env.py 未アップロード時のフォールバック（従来のコード）
    ee.Authenticate()
    try:
        ee.Initialize()
    except Exception:
        ee.Initialize(project='ee-yourproject')  # 自分のプロジェクトIDに置き換え


### ✅ 検証ポイント①
AIが生成したコードを実行する前に、以下の点を確認しよう。

1. バンド名は正しいか？（B4 = Red, B8 = NIR）
2. 雲フィルタのプロパティ名は正しいか？（`CLOUDY_PIXEL_PERCENTAGE`）
3. 緯度経度の順序は正しいか？（経度, 緯度 の順）

In [ ]:
# 🤖 AIが生成したコード（例） — 実際に実行してみよう

roi = ee.Geometry.Rectangle([139.9, 37.2, 140.6, 37.8])

collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2024-06-01', '2024-06-30')
    .filterBounds(roi)
    .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 20)
)

print(f'該当画像数: {collection.size().getInfo()}')

image = collection.sort('CLOUDY_PIXEL_PERCENTAGE').first()
print(f'選択画像: {image.id().getInfo()}')
print(f'撮影日時: {image.date().format("YYYY-MM-dd").getInfo()}')

In [ ]:
# NDVI計算
ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDVI の統計情報を確認
stats = ndvi.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=100,
    bestEffort=True
)
print(f'NDVI 範囲: {stats.getInfo()}')

### ✅ 検証ポイント②
NDVIの値域を確認しよう。

| 結果 | 判定 |
|------|------|
| 最小値が **-1以上**、最大値が **1以下** | ✅ 正常 |
| 範囲外の値がある | ❌ 計算式かバンド指定に誤りがある。AIに修正を依頼しよう |

> 💬 修正プロンプト例:
> 「NDVIの値が[-1, 1]の範囲を超えています。バンド指定が間違っている可能性があります。Sentinel-2 L2AのRedバンドとNIRバンドのインデックスを確認して修正してください。」

In [ ]:
# geemap で可視化
m = geemap.Map()

vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2000}
m.add_layer(image, vis_rgb, 'True Color (RGB)')

vis_ndvi = {'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
m.add_layer(ndvi, vis_ndvi, 'NDVI')

m.center_object(roi)
m

---
## Part B: GEE + Sentinel-1 SAR

### シナリオ
同じエリアのSAR画像（Sentinel-1 VH偏波）を取得して、光学との違いを確認したい。

### 💬 Gemini/ChatGPTへのプロンプト例
```
あなたはGEEのPython APIの専門家です。
Sentinel-1 SAR GRDデータから後方散乱係数（VH偏波）を取得するコードを書いてください。

- 範囲: ee.Geometry.Rectangle([139.9, 37.2, 140.6, 37.8])
- 期間: 2024-06-01 〜 2024-06-30
- 観測方向: ASCENDING
- 偏波: VH
- geemapで表示（グレースケール）
- 後方散乱係数の範囲も表示
```

In [ ]:
# 🤖 AIが生成したコード（例） — 改善版（空コレクション対応）

sar_all = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterDate('2024-06-01', '2024-06-30')
    .filterBounds(roi)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
)

asc = sar_all.filter(ee.Filter.eq('orbitProperties_pass', 'ASCENDING'))
desc = sar_all.filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
print(f'ASCENDING: {asc.size().getInfo()} 件, DESCENDING: {desc.size().getInfo()} 件')

sar_collection = desc if desc.size().getInfo() > 0 else asc
count = sar_collection.size().getInfo()
print(f'該当SAR画像数: {count}')

if count > 0:
    sar_image = sar_collection.first()
    print(f'選択画像: {sar_image.id().getInfo()}')
    vh = sar_image.select('VH')
    vh_db = ee.Image.constant(10).multiply(vh.log10()).rename('VH_db')
    sar_stats = vh_db.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=100,
        bestEffort=True
    )
    print(f'SAR VH (dB) 範囲: {sar_stats.getInfo()}')
else:
    print('該当するSAR画像がありません。期間や範囲を見直してください。')
    sar_image = None

### ✅ 検証ポイント

| チェック項目 | 判定方法 |
|------------|---------|
| 偏波は正しいか？ | Sentinel-1 GRDのデフォルトはVV+VH。HH偏波は使用不可 |
| dB値の範囲は？ | 農地のVH後方散乱は **-25〜-10dB** 程度が目安。極端に異なる値なら要確認 |
| 観測方向は適切か？ | ASCENDING / DESCENDING の両方を試すとよい |

In [ ]:
# SARの表示
if sar_image is not None:
    m2 = geemap.Map()
    m2.add_layer(sar_image, {'bands': 'VH', 'min': -25, 'max': 0}, 'VH (dB)')
    m2.center_object(roi)
    m2
else:
    print('SAR画像がないため表示をスキップします')

---
## Part C: JAXA Earth API で標高データを取得

### シナリオ
JAXA Earth APIを使って、同じエリアのALOS AW3D30標高データを取得し、可視化する。

### 💬 Gemini/ChatGPTへのプロンプト例
```
あなたはJAXA Earth API（jaxa-earth）のPythonの専門家です。
以下のコードを生成してください。

- jaxa-earth ライブラリを使ってALOS AW3D30 DSMを取得
- 範囲: 東経139.9-140.6, 北緯37.2-37.8
- 取得したデータをmatplotlibで表示
- 標高の最小値・最大値・平均値を計算
```

In [ ]:
# JAXA Earth API のインストール
!pip install jaxa-earth

In [ ]:
# 🤖 AIが生成したコード（例）
from jaxa.earth import je
import matplotlib.pyplot as plt
import numpy as np

bbox = [139.9, 37.2, 140.6, 37.8]

# ALOS AW3D30 DSM を取得
collection = 'JAXA.EORC_ALOS.PRISM_AW3D30.v3.2_global'
band = 'DSM'
dlim = ['2021-02-01T00:00:00', '2021-02-01T00:00:00']
ppu = 360

data = (
    je.ImageCollection(collection=collection, ssl_verify=True)
    .filter_date(dlim=dlim)
    .filter_resolution(ppu=ppu)
    .filter_bounds(bbox=bbox)
    .select(band=band, dlim=dlim)
)
img_data = je.ImageProcess(data).show_images()

print(f'データ形状: {img_data[0]["DSM"].shape}')
dem_array = img_data[0]['DSM']
print(f'標高 最小: {np.nanmin(dem_array):.1f}m')
print(f'標高 最大: {np.nanmax(dem_array):.1f}m')
print(f'標高 平均: {np.nanmean(dem_array):.1f}m')

plt.figure(figsize=(10, 6))
plt.imshow(dem_array, cmap='terrain', vmin=0, vmax=500)
plt.colorbar(label='標高 (m)')
plt.title('ALOS AW3D30 DSM — 福島県浜通り')
plt.show()

### ✅ 検証ポイント

| チェック項目 | 判定方法 |
|------------|---------|
| 標高値は妥当か？ | 福島県浜通りは沿岸部=0m, 阿武隈山地=200〜500m程度。極端な値がないか確認 |
| 範囲が正しいか？ | 期待したエリアが表示されているか確認 |
| データがない場合 | 範囲が広すぎる可能性。`bbox` を狭めて再試行 |

---
## Part D: AIを使ったデバッグ練習

### シナリオ
以下のコードには**意図的にエラーや問題が含まれている**。AIを使って修正してみよう。

### 💬 デバッグの進め方
1. 下のセルを実行してエラーを出す
2. エラーメッセージをコピーしてAIに貼り付ける
3. AIの修正案を適用する
4. 再度実行して直ったか確認する

In [ ]:
# このコードにはバグがあります — 実行してエラーを確認しよう

roi_wrong = ee.Geometry.Rectangle([37.2, 139.9, 37.8, 140.6])  # 緯度経度が逆

collection_bug = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2024-06-01', '2024-06-30')
    .filterBounds(roi_wrong)
    .filterMetadata('CLOUD_COVER', 'less_than', 20)  # 存在しないプロパティ名
)

print(collection_bug.size().getInfo())

### 💬 AIに送るプロンプト例

```
以下のエラーが発生しました。原因と修正方法を教えてください。

コード:
（バグを含むコードを貼り付け）

エラーメッセージ:
（表示されたエラーを貼り付け）\n
    ```\n

---

### 💡 練習問題（任意）

AIに以下のタスクを依頼して、生成されたコードの正しさを検証してみよう。

1. 「NDVIの時系列グラフを描くコードを書いて」
2. 「農地ポリゴン内のNDVI平均値を計算して」
3. 「Sentinel-1とSentinel-2を重ねて表示するコードを書いて」

各コードについて、以下の観点で検証すること。

- NDVIの値域は -1〜1 か？
- バンド指定は正しいか？
- 雲フィルタは適切か？
- 座標系は合っているか？
- 季節・期間は解析目的に合っているか？

---
## まとめ

### AIコーディングのベストプラクティス

1. **具体的に指示する** — 「コードを書いて」ではなく「Sentinel-2のNDVIをGEEで計算するPythonコードを書いて」
2. **実行して検証する** — AIの出力をそのまま信じず、必ず実行して値の範囲を確認する
3. **エラーはそのまま貼る** — エラーメッセージをAIにそのままコピペするのが最速のデバッグ方法
4. **理由を説明させる** — 「なぜそう書いたか」をAIに説明させると、自分の理解が深まる
5. **間違いを記録する** — AIが間違えた例は演習ログに残そう。それがあなたの学びになる

### 参考リンク

- [AIプロンプトガイド](../resources/ai-coding-guide.md) — より詳しいプロンプト集
- [AIセットアップ手順](../resources/ai-setup.md) — 各AIツールの導入方法
- [GEEチュートリアル](00_example_gee.ipynb) — GEEの基本操作